<a href="https://colab.research.google.com/github/MaLith344/Statistical-Learning-e21344/blob/main/E_21_344_Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

## Answer



**1. Prior Belief Boundaries**

Before data collection begins, the initial prior distribution over the remaining stiffness efficiency factor $\Theta \in (0, 1]$ is modeled using a Beta distribution: $\Theta \sim \text{Beta}(8, 1.5)$.

The expected value of a Beta-distributed random variable is calculated analytically as:


$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.8421$$

Suitability Analysis:
The $\text{Beta}(8, 1.5)$ distribution is highly skewed to the right, placing the vast majority of its probability mass near $1.0$. This mathematically encodes the initial engineering assumption that the structural component is highly likely to be pristine and healthy before inspection. Simultaneously, the extended left tail realistically accounts for the physical possibility of minor pre-existing manufacturing defects or early-stage, undetected fatigue, rather than naively assigning $100\%$ certainty to perfect health.

**2. Structural Likelihood Formulation**

The experimental sensor measurement $y_k$ is modeled as $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$, where $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$. By taking the natural logarithm of both sides, we yield:


$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k$$

Because $\epsilon_k$ is normally distributed, the measurement $Y_k$ follows a log-normal distribution conditional on $\Theta = \theta$. The mathematical likelihood contribution of a single continuous observation $y_k$ given the true stiffness factor $\theta$ is the log-normal probability density function:


$$L(y_k \mid \theta) = f_{Y_k \mid \Theta}(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln y_k - \ln(\theta K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$

Assuming that sequential sensor noise terms across different time steps are conditionally independent given $\Theta = \theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is the product of the individual step likelihoods:


$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \left( \frac{1}{\sigma \sqrt{2\pi}} \right)^k \left( \prod_{i=1}^k \frac{1}{y_i} \right) \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^k \left(\ln y_i - \ln(\theta K_{\text{nominal}})\right)^2 \right)$$

**3. Mathematical Formulation of the Non-Conjugate Grid Update**

An exact closed-form analytical solution for the posterior density does not exist because the Beta prior (governed by polynomial terms $\theta^{\alpha-1}(1-\theta)^{\beta-1}$) and the Log-normal likelihood (governed by exponential terms of $\ln(\theta)$) are not conjugate pairs. Their product cannot be algebraically rearranged into the standard functional form of any recognized parametric probability distribution family.

Consequently, the recursive relationship for the continuous posterior density at step $k$, up to a proportionality constant, must be expressed via Bayes' rule as:


$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

**4. Running Point Estimates**

Because a closed-form parametric formula is unavailable, running point estimators must be derived via definite numerical integration over the bounded physical domain $(0, 1]$.

Running Posterior Mean ($\widehat{\theta}_{\text{Bayes}}^{(k)}$): Under a squared-error loss function, the optimal Bayes estimate is the expected value of the current posterior distribution:


$$\widehat{\theta}_{\text{Bayes}}^{(k)} = \int_{0}^{1} \theta \, f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

Running Maximum A Posteriori ($\widehat{\theta}_{\text{MAP}}^{(k)}$): The most probable structural state corresponds to the peak (mode) of the posterior density over the bounded domain:


$$\widehat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

**5. Algorithmic Grid Approximation and Normalization**

To maintain and sequentially update this distribution computationally, the continuous parameter space is evaluated over a fine discrete grid.

Step-by-Step Numerical Procedure:

Grid Discretization: Define an array of $M$ equally spaced discrete values over the boundary limit $[\theta_{\min}, \theta_{\max}]$ (e.g., restricting the lower bound to $0.01$ to avoid physically impossible $\log(0)$ evaluations). Let this grid be $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$.

Prior Initialization: Evaluate the Beta PDF across $\boldsymbol{\theta}$ to form the initial probability array $\mathbf{P}_0$. Normalize it using the composite trapezoidal rule so the total area equals exactly $1$:


$$Z_0 = \text{trapezoid}(\mathbf{P}_0, \boldsymbol{\theta}), \qquad \mathbf{P}_0 \leftarrow \frac{\mathbf{P}_0}{Z_0}$$

Sequential Likelihood Updating: For each incoming sensor reading $y_k$:

Compute the likelihood array $L_k(\boldsymbol{\theta})$ by evaluating the log-normal PDF across all $\theta_m \in \boldsymbol{\theta}$.

Multiply element-wise with the previous step's posterior to get the unnormalized posterior: $\tilde{\mathbf{P}}_k = \mathbf{P}_{k-1} \odot L_k(\boldsymbol{\theta})$.

Trapezoidal Normalization: Calculate the normalization constant $Z_k$ by integrating the unnormalized posterior array over the grid:


$$Z_k = \sum_{m=1}^{M-1} \frac{\tilde{P}_k(\theta_m) + \tilde{P}_k(\theta_{m+1})}{2} \Delta\theta$$

Final Posterior Constraint: Divide the unnormalized array by $Z_k$ to enforce the probability axiom (area under curve = 1):


$$\mathbf{P}_k = \frac{\tilde{\mathbf{P}}_k}{Z_k}$$

**6. Performance Tracking and Degradation Convergence Analysis**

Convergence Behavior:
Based on the underlying degradation physics and as demonstrated in the simulation, it typically requires only about 3 to 4 continuous sensor readings for the Bayesian system to overcome the initially optimistic prior ($\theta \approx 0.84$) and firmly isolate the true damage state at $\theta_{\text{true}} = 0.68$. Because the measurement noise is relatively low ($\sigma = 0.15$), each sequential likelihood curve acts as a highly informative constraint, aggressively dragging the probability mass away from the baseline assumption and toward the experimental reality.

Safety Threshold Implications:
As the system reaches $n = 15$ measurements, the massive narrowing of the posterior density curves reflects a near-total reduction in epistemic uncertainty. In an engineering context, this sharp posterior implies that the system is highly confident in the exact magnitude of the structural degradation. This statistical certainty is critical because it allows automated SHM systems to trigger rigid safety protocols (e.g., issuing a grounding order if the cumulative probability $P(\Theta < 0.70) > 0.99$) with minimal risk of triggering false positive alarms.

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

def run_shm_simulation():
    """
    Simulates a sequence of Structural Health Monitoring (SHM) sensor readings,
    updates the Bayesian posterior density over a bounded discrete grid,
    and visualizes the degradation convergence using Plotly.
    """

    # 1. Parameter Initialization
    n_steps = 15               # Number of sequential sensor readings
    theta_true = 0.68          # The true, hidden remaining stiffness factor (damage state)
    K_nom = 50.0               # Nominal baseline stiffness (kN/mm)
    sigma = 0.15               # Sensor noise standard deviation (log-space)

    # Beta Prior parameters for the initial healthy assumption
    alpha_prior = 8.0
    beta_prior = 1.5

    # Boundary constraints enforced via grid limits
    # (Starting slightly above 0 to prevent log(0) domain errors)
    theta_grid = np.linspace(0.01, 1.0, 1000)

    # 2. Sensor Data Simulation
    np.random.seed(42)  # Set seed for reproducible simulation results

    # The true mean of the underlying log-normal physical model
    mu_true = np.log(theta_true * K_nom)

    # Draw noisy readings from the physical log-normal physics model
    y_measurements = np.random.lognormal(mean=mu_true, sigma=sigma, size=n_steps)

    # 3. Grid Approximation Setup
    # Initialize the prior using the Beta distribution PDF
    prior_pdf = stats.beta.pdf(theta_grid, alpha_prior, beta_prior)
    # Ensure perfectly normalized area = 1.0 over the bounded grid
    prior_pdf /= np.trapezoid(prior_pdf, theta_grid)

    # Storage dictionaries and lists for timeline analysis and plotting
    density_history = {0: prior_pdf.copy()}
    bayes_estimates = [np.trapezoid(theta_grid * prior_pdf, theta_grid)]
    map_estimates = [theta_grid[np.argmax(prior_pdf)]]

    current_posterior = prior_pdf.copy()
    milestones = [1, 2, 5, 10, 15]  # Specific steps to track the full density curves

    # 4. Recursive Sequential Update
    for k, y_k in enumerate(y_measurements, start=1):
        # Log-normal likelihood evaluation for the incoming sensor reading y_k
        # Note: scale parameter in scipy corresponds to exp(mu) = theta * K_nom
        likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=theta_grid * K_nom)

        # Bayes Update: Multiply Prior by Likelihood
        unnormalized_posterior = current_posterior * likelihood

        # Trapezoidal Normalization: Divide by the area under the unnormalized curve
        Z_k = np.trapezoid(unnormalized_posterior, theta_grid)
        current_posterior = unnormalized_posterior / Z_k

        # Store milestone density curves for the first plot
        if k in milestones:
            density_history[k] = current_posterior.copy()

        # Point Estimate Evaluation for the current step
        # Bayes Estimate (Mean): integral(theta * p(theta) d_theta)
        theta_bayes = np.trapezoid(theta_grid * current_posterior, theta_grid)
        bayes_estimates.append(theta_bayes)

        # MAP Estimate (Mode): argmax(p(theta))
        theta_map = theta_grid[np.argmax(current_posterior)]
        map_estimates.append(theta_map)

    # 5. Visualization Generation

    # Plot 1: Posterior Density Progression
    fig1 = go.Figure()
    colors = ['#C0C0C0', '#900C3F', '#C70039', '#FF5733', '#FFC300', '#DAF7A6']
    labels = ['Initial Prior (k=0)'] + [f'Posterior (k={m})' for m in milestones]

    for idx, (k_step, density) in enumerate(density_history.items()):
        fig1.add_trace(go.Scatter(
            x=theta_grid,
            y=density,
            mode='lines',
            name=labels[idx],
            line=dict(color=colors[idx], width=2 if k_step == 0 else 3),
            fill='tozeroy' if k_step == 15 else 'none'
        ))

    # Add a reference line for the true hidden damage state
    fig1.add_vline(x=theta_true, line_dash="dash", line_color="black",
                   annotation_text="True Damage State (0.68)", annotation_position="top left")

    fig1.update_layout(
        title='1. Evolution of Structural Health Posterior Density (k: 0 to 15)',
        xaxis_title='Stiffness Efficiency Factor (θ)',
        yaxis_title='Probability Density',
        template='plotly_white',
        hovermode="x unified"
    )
    fig1.show()

    # Plot 2: Estimator Convergence
    steps = np.arange(0, n_steps + 1)
    fig2 = go.Figure()

    # Plot the running Posterior Mean
    fig2.add_trace(go.Scatter(
        x=steps, y=bayes_estimates, mode='lines+markers',
        name='Bayes Estimate (Mean)', line=dict(color='blue', width=3), marker=dict(size=8)
    ))

    # Plot the running MAP Estimate
    fig2.add_trace(go.Scatter(
        x=steps, y=map_estimates, mode='lines+markers',
        name='MAP Estimate (Mode)', line=dict(color='orange', width=3), marker=dict(size=8)
    ))

    # Add a horizontal reference line for the true target state
    fig2.add_hline(y=theta_true, line_dash="dash", line_color="red", line_width=3,
                   annotation_text="True Hidden State (0.68)", annotation_position="bottom right")

    fig2.update_layout(
        title='2. Tracking Convergence of Running Point Estimators',
        xaxis_title='Sensor Inspection Step (k)',
        yaxis_title='Estimated Stiffness Factor (θ)',
        template='plotly_white',
        hovermode="x unified"
    )
    fig2.show()

if __name__ == "__main__":
    try:
        run_shm_simulation()
    except Exception as e:
        print(f"An error occurred while running the simulation: {e}")

#Sample Answer

1. Physical Likelihood Formulation

At time step $k$, a physical sensor records a continuous measurement $Y_k = y_k$ (such as dynamic modal frequency, strain measurement, or vibrational response).

The measurement model is governed by a known structural forward response function $g(\theta)$ subject to additive Gaussian sensor noise $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$:

$$Y_k = g(\theta) + \epsilon_k$$

Conditional on the underlying structural integrity parameter $\Theta = \theta$, the likelihood contribution of a single observation $y_k$ at step $k$ is given by the Gaussian probability density function:

$$L(y_k \mid \theta) = f_{Y_k \mid \Theta}(y_k \mid \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{\left(y_k - g(\theta)\right)^2}{2\sigma^2} \right)$$

---

2. Sequential Likelihood and Joint History

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)^T$ represent the vector of sequential sensor observations gathered up to step $k$.

Assuming that sensor noise terms across consecutive time steps are conditionally independent given $\Theta = \theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \frac{1}{(2\pi\sigma^2)^{k/2}} \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^k \left(y_i - g(\theta)\right)^2 \right)$$

---

3. Mathematical Formulation of Bounded Recursive Updates

Before observing measurements, the platform initializes a prior density function $f_{\Theta}^{(0)}(\theta)$ over the physically bounded interval $[\theta_{\min}, \theta_{\max}]$ (e.g., a uniform distribution $U(\theta_{\min}, \theta_{\max})$ indicating initial uninformative uncertainty).

Under a sequential Bayesian updating framework, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$. The running posterior density function $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ is recursively updated via Bayes' Theorem:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{\theta_{\min}}^{\theta_{\max}} L(y_k \mid s) \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)}) \, ds}$$

*Definition of Key Components*:

* **$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$**: The prior density at step $k$ inherited directly from the previous state $k-1$.
* **$L(y_k \mid \theta)$**: The likelihood contribution of the incoming real-time sensor reading $y_k$.
* **Denominator (Normalizing Constant $Z_k$)**: Integrates the product of likelihood and prior across the bounded domain $[\theta_{\min}, \theta_{\max}]$ to ensure the total area under the density curve equals $1$.

---

4. Running Point Estimators

From the running posterior distribution $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, two primary estimators track structural health at step $k$:

a. Running Posterior Mean ($\widehat{\theta}_{\text{Bayes}}^{(k)}$)
Under a squared-error loss function, the optimal point estimate is the expected value of the current bounded posterior distribution:

$$\widehat{\theta}_{\text{Bayes}}^{(k)} = \mathbb{E}\left[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}\right] = \int_{\theta_{\min}}^{\theta_{\max}} \theta \, f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

b.. Running Maximum A Posteriori ($\widehat{\theta}_{\text{MAP}}^{(k)}$)
The most probable structural state corresponds to the peak (mode) of the current posterior density over the bounded domain:

$$\widehat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in [\theta_{\min}, \theta_{\max}]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

---

5. Numerical Implementation via Bounded Grid Discretization

Since non-linear structural response functions $g(\theta)$ generally lack closed-form analytical conjugate solutions, the system maintains the posterior on a fine discrete grid across the bounded physical domain $[\theta_{\min}, \theta_{\max}]$.

*Algorithmic Procedure*:

a. **Grid Setup:** Define $M$ equally spaced grid points over $[\theta_{\min}, \theta_{\max}]$:
   $$\theta_m = \theta_{\min} + (m-1)\Delta\theta, \quad \text{where } \Delta\theta = \frac{\theta_{\max} - \theta_{\min}}{M-1}, \quad m = 1, 2, \dots, M$$

b. **Prior Initialization:** Evaluate initial prior values across the grid array $\mathbf{P}_0 = [P_0(\theta_1), \dots, P_0(\theta_M)]$ and normalize using numerical integration (e.g., composite trapezoidal rule):
   $$Z_0 = \text{trapezoid}(\mathbf{P}_0, \boldsymbol{\theta}), \quad \mathbf{P}_0 \leftarrow \frac{\mathbf{P}_0}{Z_0}$$

c. **Sequential Updating & Normalization (at step $k$):**
   * Compute unnormalized posterior array: $\tilde{P}_k(\theta_m) = P_{k-1}(\theta_m) \times L(y_k \mid \theta_m)$
   * Compute normalizing factor: $Z_k = \sum_{m=1}^{M-1} \frac{\tilde{P}_k(\theta_m) + \tilde{P}_k(\theta_{m+1})}{2} \Delta\theta$
   * Normalize: $P_k(\theta_m) = \frac{\tilde{P}_k(\theta_m)}{Z_k}$

d. **Point Estimation Evaluation:**
   * **Bayes Estimate:** $\widehat{\theta}_{\text{Bayes}}^{(k)} \approx \text{trapezoid}(\boldsymbol{\theta} \odot \mathbf{P}_k, \boldsymbol{\theta})$
   * **MAP Estimate:** $\widehat{\theta}_{\text{MAP}}^{(k)} = \theta_{m^*}, \quad \text{where } m^* = \arg\max_{m} P_k(\theta_m)$

---

6. Dynamic Mechanics & Convergence Interpretation

* **Variance Reduction:** As $k$ increases, repeated noisy sensor measurements progressively attenuate likelihood ambiguity, tightening the posterior distribution variance around the true structural state $\theta_{\text{true}}$.
* **Physical Boundary Enforcement:** The bounded grid framework strictly guarantees that probability mass outside $[\theta_{\min}, \theta_{\max}]$ is zero, avoiding physically unfeasible state estimates (such as negative stiffness or over-100% health).
* **Sensor Noise vs. Tracking Sensitivity:** Lower measurement noise $\sigma$ narrows the single-step likelihood curve $L(y_k \mid \theta)$, allowing rapid posterior convergence with fewer sensor readings.

In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set random seed for reproducibility
np.random.seed(24)

# =====================================================================
# CONFIGURATION & PARAMETERS
# =====================================================================
theta_true = 0.68       # True remaining stiffness efficiency of the beam (68%)
K_nominal = 50.0        # Nominal baseline stiffness of the pristine structure (kN/mm)
sigma = 0.15            # Sensor noise standard deviation (log-space)
n_sensor_readings = 15  # Timeline steps

# 1. Define a fine grid over the physical boundary [0.01, 1.0]
theta_grid = np.linspace(0.01, 1.0, 500)

# 2. Initialize Prior: Bounded Beta distribution reflecting an initially healthy beam
# centered heavily near 0.95-1.0
current_posterior = stats.beta.pdf(theta_grid, a=8, b=1.5)
# Normalize initial prior
current_posterior /= np.trapezoid(current_posterior, theta_grid)

# Steps milestone tracking for plotting curves
milestones = [0, 1, 2, 5, 10, 15]

# Create Figure
fig = go.Figure()

# Plot Initial Prior State
fig.add_trace(go.Scatter(
    x=theta_grid, y=current_posterior, mode='lines',
    name='Prior State: Structural Health Assumed Healthy',
    line=dict(dash='dash', width=2.5, color='gray')
))

# =====================================================================
# SEQUENTIAL BAYESIAN MONITORING LOOP
# =====================================================================
for k in range(1, n_sensor_readings + 1):
    # Simulate a noisy structural sensor reading from log-normal physics
    noise = np.random.normal(0, sigma)
    y_k = (theta_true * K_nominal) * np.exp(noise)

    # Calculate Log-Normal Likelihood curve across the structural theta grid
    # Expected value for any grid point is: grid_point * K_nominal
    expected_K = theta_grid * K_nominal
    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=expected_K)

    # Running Update: Posterior Proportional to Prior * Likelihood
    current_posterior = current_posterior * likelihood

    # Numerical Normalization via Trapezoidal rule
    integral = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= integral

    # Capture structural health density profile at milestones
    if k in milestones:
        fig.add_trace(go.Scatter(
            x=theta_grid, y=current_posterior, mode='lines',
            name=f"Step {k}: Post-Sensor Reading (Observed K={y_k:.2f})",
            line=dict(width=2)
        ))

# =====================================================================
# VISUALIZE STRUCTURAL DEGRADATION TRACKING
# =====================================================================
fig.add_vline(
    x=theta_true, line_dash="dot", line_color="red", line_width=2.5,
    annotation_text=f"True Structural Degradation State ({theta_true})",
    annotation_position="top left"
)

fig.update_layout(
    title={
        'text': "Structural Health Monitoring: Bounded Bayesian Parameter Updating",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Remaining Structural Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density (Confidence level of damage)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="left", x=0.02,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

fig.show()